In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bismasajjad/global-ai-job-market-and-salary-trends-2025")

print("Path to dataset files:", path)



Path to dataset files: /Users/michaelchung/.cache/kagglehub/datasets/bismasajjad/global-ai-job-market-and-salary-trends-2025/versions/2


# What does the AI job market reward most?
**The role you hold, the experience you bring, or the skills you have?**

Each chart below answers one small question. Together they build to the answer. Hover over any chart to see exact numbers.

> **About the data:** The [Kaggle page](https://www.kaggle.com/datasets/bismasajjad/global-ai-job-market-and-salary-trends-2025) describes this as *"a synthetic dataset created for educational purposes"*. The 15,000 postings were generated by an algorithm, not collected from real job boards. Read the patterns as how the dataset was built, not as real-world facts. When a chart shows "no difference", that is a finding too.

In [2]:
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv(os.path.join(path, "ai_job_dataset.csv"))
df["posting_date"] = pd.to_datetime(df["posting_date"])
df["month"] = df["posting_date"].dt.to_period("M").dt.to_timestamp()

EXP = {"EN": "Entry", "MI": "Mid", "SE": "Senior", "EX": "Executive"}
REMOTE = {0: "On-site", 50: "Hybrid", 100: "Fully remote"}
df["experience"] = pd.Categorical(df["experience_level"].map(EXP), list(EXP.values()), ordered=True)
df["work_mode"] = pd.Categorical(df["remote_ratio"].map(REMOTE), list(REMOTE.values()), ordered=True)

# One visual style for every figure: one highlight colour, everything else grey
BLUE, GREY, INK, MUTED = "#2a78d6", "#c3c2b7", "#0b0b0b", "#52514e"
EXP_COLORS = dict(zip(EXP.values(), ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]))  # light -> dark = junior -> senior

def style(fig, title, subtitle, height=480):
    fig.update_layout(
        title=dict(text=f"<b>{title}</b><br><span style='font-size:13px;color:{MUTED}'>{subtitle}</span>", x=0, xanchor="left"),
        font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", size=13, color=INK),
        plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb", height=height,
        margin=dict(l=10, r=30, t=90, b=40), showlegend=False,
        hoverlabel=dict(bgcolor="white", font_size=13),
    )
    fig.update_xaxes(gridcolor="#e1e0d9", zeroline=False, linecolor="#c3c2b7")
    fig.update_yaxes(gridcolor="#e1e0d9", zeroline=False, linecolor="#c3c2b7")
    return fig

def money(v):
    return f"${v/1000:,.0f}k"

def ranked_bar(by, title, subtitle, height=560, start=0):
    """Horizontal bar of median salary per group, best group highlighted,
    with a dropdown to filter by experience level."""
    def series(sub):
        s = sub.groupby(by)["salary_usd"].median().sort_values()
        colors = [GREY] * (len(s) - 1) + [BLUE]
        return s, colors
    views = {"All experience levels": df, **{f"{e} level only": df[df["experience"] == e] for e in EXP.values()}}
    s, c = series(list(views.values())[start])
    fig = go.Figure(go.Bar(x=s.values, y=s.index, orientation="h", marker_color=c,
                           text=[money(v) for v in s.values], textposition="outside", cliponaxis=False,
                           hovertemplate="<b>%{y}</b><br>Median salary: %{x:$,.0f}<extra></extra>"))
    buttons = []
    for label, sub in views.items():
        s, c = series(sub)
        buttons.append(dict(label=label, method="update",
                            args=[{"x": [s.values], "y": [s.index], "marker.color": [c],
                                   "text": [[money(v) for v in s.values]]}]))
    fig.update_layout(updatemenus=[dict(buttons=buttons, active=start, x=1, xanchor="right", y=1.12, yanchor="bottom",
                                        bgcolor="white", bordercolor=GREY)])
    fig.update_xaxes(title="Median salary (USD)", tickprefix="$", tickformat="~s")
    return style(fig, title, subtitle, height)

## 1. Experience
### Q1.1 How does pay change as people gain experience?
Each box shows the middle half of salaries at that level. The line inside is the typical (median) salary.

In [3]:
fig = px.box(df, x="experience", y="salary_usd", color="experience", color_discrete_map=EXP_COLORS,
             category_orders={"experience": list(EXP.values())}, points=False)
fig.update_traces(hovertemplate="%{y:$,.0f}<extra></extra>")
med = df.groupby("experience", observed=True)["salary_usd"].median()
for lvl, v in med.items():
    fig.add_annotation(x=lvl, y=v, text=f"<b>{money(v)}</b>", showarrow=False, xshift=58, font_color=INK)
fig.update_yaxes(title="Salary (USD)", tickprefix="$", tickformat="~s")
fig.update_xaxes(title=None)
style(fig, f"Executives earn about {med['Executive']/med['Entry']:.0f}x what entry-level workers earn",
      "Salary by experience level. Typical salary rises at every step.").show()

### Q1.2 For people just starting out, which job title pays best?
Use the dropdown to compare other experience levels.

In [4]:
entry = df[df["experience"] == "Entry"].groupby("job_title")["salary_usd"].median()
top = entry.idxmax()
fig = ranked_bar("job_title",
    f"At entry level, {top} pays the most, but only by a little",
    f"The gap between the best- and worst-paid entry-level title is just {money(entry.max() - entry.min())}.",
    start=1).show()  # start on the entry-level view

## 2. Where and how you work
### Q2.1 Which countries pay AI workers the most?
Use the dropdown to check that the ranking holds at every experience level.

In [5]:
by_country = df.groupby("company_location")["salary_usd"].median()
ranked_bar("company_location",
    f"{by_country.idxmax()} pays roughly double what {by_country.idxmin()} pays",
    "Median salary by company location (USD).", height=620).show()

### Q2.2 Does working remotely change what you earn?

In [6]:
fig = px.box(df, x="work_mode", y="salary_usd", category_orders={"work_mode": list(REMOTE.values())}, points=False)
fig.update_traces(marker_color=BLUE, hovertemplate="%{y:$,.0f}<extra></extra>")
fig.update_yaxes(title="Salary (USD)", tickprefix="$", tickformat="~s")
fig.update_xaxes(title=None)
style(fig, "Remote, hybrid and on-site jobs pay almost the same",
      "Salary range by work arrangement. The boxes are nearly identical.").show()

## 3. Skills
### Q3.1 Which skills do employers ask for most?

In [7]:
skills = df["required_skills"].str.get_dummies(sep=", ")
freq = skills.sum().sort_values()
share = freq / len(df)
fig = go.Figure(go.Bar(x=freq.values, y=freq.index, orientation="h",
                       marker_color=[GREY] * (len(freq) - 3) + [BLUE] * 3,
                       customdata=share.values, text=[f"{v:.0%}" for v in share.values], textposition="outside", cliponaxis=False,
                       hovertemplate="<b>%{y}</b><br>%{x:,} postings (%{customdata:.0%} of all jobs)<extra></extra>"))
fig.update_xaxes(title="Number of job postings")
style(fig, f"{freq.index[-1]} is the most requested skill, listed in {share.iloc[-1]:.0%} of postings",
      "Number of postings that list each skill.", height=640).show()

### Q3.2 Do jobs that need a particular skill pay more?
Each dot shows how much more (or less) the typical job that lists that skill pays than the typical job that doesn't.

In [8]:
premium = pd.Series({s: df.loc[skills[s] == 1, "salary_usd"].median() - df.loc[skills[s] == 0, "salary_usd"].median()
                     for s in skills}).sort_values()
colors = [BLUE if v > 0 else GREY for v in premium.values]
fig = go.Figure()
for skill, v, c in zip(premium.index, premium.values, colors):  # lollipop stems
    fig.add_shape(type="line", x0=0, x1=v, y0=skill, y1=skill, line=dict(color=c, width=2))
fig.add_trace(go.Scatter(x=premium.values, y=premium.index, mode="markers", marker=dict(size=10, color=colors),
                         customdata=freq.reindex(premium.index).values,
                         hovertemplate="<b>%{y}</b><br>%{x:+$,.0f} vs. jobs without it<br>%{customdata:,} postings<extra></extra>"))
fig.add_vline(x=0, line_color=MUTED, line_width=1)
fig.update_xaxes(title="Difference in median salary (USD)", tickprefix="$", tickformat="~s", range=[-12000, 12000])
style(fig, "No single skill moves pay by more than about $4,000",
      "Median salary for jobs that list the skill, minus jobs that don't. Blue means the skill pays more.", height=640).show()

## 4. Over time
### Q4.1 Is hiring growing or shrinking month to month?
Drag the slider under the chart to zoom into a period.

In [9]:
monthly = df.groupby("month").size()
fig = go.Figure(go.Scatter(x=monthly.index, y=monthly.values, mode="lines+markers", line=dict(color=BLUE, width=2),
                           marker=dict(size=8), hovertemplate="%{x|%B %Y}<br>%{y:,} postings<extra></extra>"))
fig.update_yaxes(title="Job postings", rangemode="tozero")
fig.update_xaxes(title=None, rangeslider=dict(visible=True, thickness=0.08), dtick="M2", tickformat="%b<br>%Y")
style(fig, "Hiring is steady: about 940 new postings every month",
      f"Postings per month, {monthly.index.min():%b %Y} to {monthly.index.max():%b %Y}. No clear upward or downward trend.").show()

### Q4.2 Have any roles had unusual pay spikes or drops?
Each square compares a role's typical salary that month with its usual level. Darker blue means higher than usual and darker red means lower. Squares marked with a dot are unusual: more than 2 standard deviations from that role's normal level.

In [10]:
tbl = df.pivot_table(index="job_title", columns="month", values="salary_usd", aggfunc="median")
pct = tbl.div(tbl.median(axis=1), axis=0) - 1
z = tbl.sub(tbl.mean(axis=1), axis=0).div(tbl.std(axis=1), axis=0)
order = pct.abs().max(axis=1).sort_values().index  # most volatile roles at the top
pct, z, tbl = pct.loc[order], z.loc[order], tbl.loc[order]
fig = go.Figure(go.Heatmap(
    z=pct.values, x=pct.columns, y=pct.index, zmid=0, zmin=-0.3, zmax=0.3,
    colorscale=[[0, "#c0392b"], [0.5, "#f0efec"], [1, "#1c5cab"]], xgap=2, ygap=2,
    customdata=tbl.values, colorbar=dict(title="vs. usual", tickformat="+.0%", thickness=12),
    hovertemplate="<b>%{y}</b>, %{x|%b %Y}<br>Median salary %{customdata:$,.0f}<br>%{z:+.0%} vs. usual<extra></extra>"))
r, c = (z.abs() > 2).values.nonzero()
fig.add_trace(go.Scatter(x=pct.columns[c], y=pct.index[r], mode="markers", hoverinfo="skip",
                         marker=dict(symbol="circle", size=7, color=INK)))
fig.update_xaxes(dtick="M2", tickformat="%b<br>%Y", showgrid=False)
fig.update_yaxes(showgrid=False)
style(fig, f"Only {len(r)} of {z.size} role-months stand out, fewer than random chance alone would produce",
      f"Monthly median salary per role vs. its usual level. Dots mark months more than 2 SD from normal (about {round(z.size * 0.046)} expected by chance).", height=680).show()

## The answer
How much does typical pay change between the best- and worst-paying group for each factor?

In [11]:
gap = lambda s: s.max() - s.min()
factors = pd.Series({
    "Experience level": gap(df.groupby("experience", observed=True)["salary_usd"].median()),
    "Country": gap(by_country),
    "Job title": gap(df.groupby("job_title")["salary_usd"].median()),
    "Skills (best vs. worst skill)": gap(premium),
    "Remote vs. on-site": gap(df.groupby("work_mode", observed=True)["salary_usd"].median()),
}).sort_values()
fig = go.Figure(go.Bar(x=factors.values, y=factors.index, orientation="h",
                       marker_color=[GREY] * (len(factors) - 1) + [BLUE],
                       text=[money(v) for v in factors.values], textposition="outside", cliponaxis=False,
                       hovertemplate="<b>%{y}</b><br>Pay gap: %{x:$,.0f}<extra></extra>"))
fig.update_xaxes(title="Gap in median salary between best- and worst-paying group (USD)", tickprefix="$", tickformat="~s")
style(fig, "Experience matters most. Role and skills barely change pay.",
      "Where you work comes second. Which title you hold or which skills you list changes pay by under $10k.", height=420).show()

**Takeaway:** In this (synthetic) dataset, *experience* is what the AI job market rewards most. Moving from entry level to executive roughly triples typical pay. *Location* is a distant second. Your *job title*, your *skills*, and whether you work remotely each change typical pay by only a few thousand dollars.